# User-to-Item Recommendation System using TF-IDF
This recsys uses TF-IDF similarly to the item-to-item one as it is easily interpretable and should emphasize more distinctive wine attributes of the wines.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
wines_features = pd.read_csv(
    Path("../data/processed/wines_features.csv"),
    usecols=["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity", "content_text"]
)

ratings = pd.read_csv(
    Path("../dataset/XWines_Full_21M_ratings.csv"),
    usecols=["UserID", "WineID", "Rating"],
    dtype={"UserID": "int32", "WineID": "int32", "Rating": "float32"}
)

print("wines_features:", wines_features.shape)
print("ratings:", ratings.shape)

wines_features.head()

wines_features: (100646, 8)
ratings: (21013536, 3)


,WineID,WineName,Type,Body,Acidity,Country,RegionName,content_text
0,100001,Espumante Moscatel,Sparkling,Medium-bodied,High,Brazil,Serra Gaúcha,type_sparkling elaborate_varietal_100 body_med...
1,100002,Ancellotta,Red,Medium-bodied,Medium,Brazil,Serra Gaúcha,type_red elaborate_varietal_100 body_medium_bo...
2,100003,Cabernet Sauvignon,Red,Full-bodied,High,Brazil,Serra Gaúcha,type_red elaborate_varietal_100 body_full_bodi...
3,100004,Virtus Moscato,White,Medium-bodied,Medium,Brazil,Serra Gaúcha,type_white elaborate_varietal_100 body_medium_...
4,100005,Maison de Ville Cabernet-Merlot,Red,Full-bodied,Medium,Brazil,Serra Gaúcha,type_red elaborate_assemblage_bordeaux_red_ble...


In [3]:
wines_features = wines_features.drop_duplicates(subset="WineID").copy()
wines_features = wines_features[wines_features["content_text"].notna()].copy()
wines_features = wines_features[wines_features["content_text"].str.strip() != ""].copy()

# make sure IDs match types
wines_features["WineID"] = wines_features["WineID"].astype("int32")
ratings["WineID"] = ratings["WineID"].astype("int32")

print("clean wines_features:", wines_features.shape)

clean wines_features: (100646, 8)


In [4]:
ratings = ratings.drop_duplicates(subset=["UserID", "WineID"]).copy()

POSITIVE_THRESHOLD = 4.0
MIN_POSITIVE_INTERACTIONS = 5
MAX_POSITIVE_INTERACTIONS = 200
MAX_USERS = 5000
RANDOM_STATE = 42
SAMPLED_USERS_PATH = Path("../data/results/user_to_item_sampled_users_5000.csv")

positive = ratings[ratings["Rating"] >= POSITIVE_THRESHOLD].copy()

user_counts = positive["UserID"].value_counts()

eligible_users = user_counts[
    (user_counts >= MIN_POSITIVE_INTERACTIONS) &
    (user_counts <= MAX_POSITIVE_INTERACTIONS)
].index

positive = positive[positive["UserID"].isin(eligible_users)].copy()

if SAMPLED_USERS_PATH.exists():
    sampled_users = pd.read_csv(SAMPLED_USERS_PATH)["UserID"].to_numpy()
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sampled_users = rng.choice(
        positive["UserID"].unique(),
        size=min(MAX_USERS, positive["UserID"].nunique()),
        replace=False
    )
    SAMPLED_USERS_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.Series(sampled_users).to_csv(SAMPLED_USERS_PATH, index=False, header=["UserID"])

positive = positive[positive["UserID"].isin(sampled_users)].copy()

print("users used in this run:", positive["UserID"].nunique())
print("positive interactions used:", positive.shape)

users used in this run: 5000
positive interactions used: (71243, 3)


Since we are working with a very large dataset, some of the recsys we've built crashed, so in order to test them equally we had to use only a subset of the full dataset. We decided to keep users with at least 5 and at most 200 positive interactions and then sampled 5000 users with a fixed seed.

In [5]:
TEST_FRACTION = 0.2
RANDOM_STATE = 42

train_parts = []
test_parts = []

for user_id, group in positive.groupby("UserID"):
    group = group.sample(frac=1, random_state=RANDOM_STATE)  # shuffle
    n_test = max(1, int(np.ceil(len(group) * TEST_FRACTION)))
    
    test_group = group.iloc[:n_test]
    train_group = group.iloc[n_test:]
    
    if len(train_group) == 0:
        continue
    
    train_parts.append(train_group)
    test_parts.append(test_group)

train_pos = pd.concat(train_parts, ignore_index=True)
test_pos = pd.concat(test_parts, ignore_index=True)

print("train_pos:", train_pos.shape)
print("test_pos:", test_pos.shape)
print("train users:", train_pos["UserID"].nunique())
print("test users:", test_pos["UserID"].nunique())

train_pos: (55019, 3)
test_pos: (16224, 3)
train users: 5000
test users: 5000


In [6]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(wines_features["content_text"])

print(tfidf_matrix.shape)

(100646, 93506)


In [7]:
wines_features = wines_features.reset_index(drop=True)

wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()
idx_to_wineid = pd.Series(wines_features["WineID"].values, index=wines_features.index)

print("mapped wines:", len(wineid_to_idx))

mapped wines: 100646


In [8]:
train_pos = train_pos[train_pos["WineID"].isin(wineid_to_idx.index)].copy()
test_pos = test_pos[test_pos["WineID"].isin(wineid_to_idx.index)].copy()

print("train_pos after filtering:", train_pos.shape)
print("test_pos after filtering:", test_pos.shape)

train_pos after filtering: (55019, 3)
test_pos after filtering: (16224, 3)


In [9]:
def build_user_profile(user_history):
    user_history = user_history[user_history["WineID"].isin(wineid_to_idx.index)].copy()
    
    item_indices = [wineid_to_idx[wine_id] for wine_id in user_history["WineID"]]
    weights = (user_history["Rating"] - 3.0).clip(lower=1.0).to_numpy()
    
    item_matrix = tfidf_matrix[item_indices]
    weighted_sum = item_matrix.multiply(weights.reshape(-1, 1)).sum(axis=0)
    profile = weighted_sum / weights.sum()
    
    return csr_matrix(profile)

In [10]:
user_profiles = {}
for user_id, user_history in train_pos.groupby("UserID"):
    user_profiles[user_id] = build_user_profile(user_history)

print("built profiles for users:", len(user_profiles))

built profiles for users: 5000


In [11]:
train_seen = train_pos.groupby("UserID")["WineID"].apply(set).to_dict()
test_relevant = test_pos.groupby("UserID")["WineID"].apply(set).to_dict()

eval_users = sorted(set(user_profiles.keys()) & set(test_relevant.keys()))
print("evaluation users:", len(eval_users))

evaluation users: 5000


For each user we split the positive interactions into the train and test sets so we ensure each user has some known preferences for building a profile. In order to create a profile we create a weighted average of TF-IDF wine vectors, higher ratings contribute more strongly to the final preference. 

In [12]:
def recommend_user_tfidf_ids(user_id, top_k=10):
    if user_id not in user_profiles:
        return []
    
    profile = user_profiles[user_id]
    scores = linear_kernel(profile, tfidf_matrix).flatten()
    
    seen_items = train_seen.get(user_id, set())
    ranked_indices = np.argsort(scores)[::-1]
    
    recommendations = []
    for idx in ranked_indices:
        wine_id = idx_to_wineid[idx]
        if wine_id in seen_items:
            continue
        recommendations.append(wine_id)
        if len(recommendations) == top_k:
            break
    
    return recommendations

In [13]:
def show_user_recommendations(user_id, top_k=10):
    rec_ids = recommend_user_tfidf_ids(user_id, top_k=top_k)
    
    recs = wines_features[wines_features["WineID"].isin(rec_ids)][
        ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
    ].copy()
    
    recs["rank"] = recs["WineID"].map({wine_id: i + 1 for i, wine_id in enumerate(rec_ids)})
    recs = recs.sort_values("rank").reset_index(drop=True)
    
    return recs

Recommendations are generated by ranking unseen wines according the cosine similarity with the user profile. 

In [14]:
sample_user = eval_users[0]
show_user_recommendations(sample_user, top_k=10)

,WineID,WineName,Type,Country,RegionName,Body,Acidity,rank
0,183893,Pinot Noir,Red,United States,Los Carneros,Medium-bodied,High,1
1,179214,Pinot Noir,Red,United States,Sonoma Coast,Medium-bodied,High,2
2,182346,Pinot Noir,Red,United States,Sonoma Coast,Medium-bodied,High,3
3,182100,Pinot Noir,Red,United States,Los Carneros,Medium-bodied,High,4
4,182376,Pinot Noir,Red,United States,Los Carneros,Medium-bodied,High,5
5,182722,Pinot Noir,Red,United States,Oregon,Medium-bodied,High,6
6,179669,Pinot Noir,Red,United States,Los Carneros,Medium-bodied,High,7
7,190432,Pinot Noir,Red,United States,Willamette Valley,Medium-bodied,High,8
8,181905,Pinot Noir,Red,United States,Sonoma Coast,Medium-bodied,High,9
9,180896,Pinot Noir,Red,United States,Los Carneros,Medium-bodied,High,10


The recommendations we get for the sample user are very consistent, all of them are red wines, specifically pinot noirs from the US, only differing in the region they are from, but still we can see a pattern of Los Carneros and Sonoma Coast. However, if our goal is to diversify the recommendations a little bit instead of showing always the best matches, this type of recsys might not be ideal.

In [15]:
def precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / k

def recall_at_k(relevant, recommended, k):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / len(relevant)

def dcg_at_k(relevant, recommended, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            score += 1.0 / np.log2(i + 1)
    return score

def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / ideal_dcg

def average_precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    hits = 0
    score = 0.0
    
    for i, item in enumerate(recommended_k, start=1):
        if item in relevant:
            hits += 1
            score += hits / i
    
    denom = min(len(relevant), k)
    return score / denom if denom > 0 else 0.0

def hit_rate_at_k(relevant, recommended, k):
    return float(any(item in relevant for item in recommended[:k]))

def reciprocal_rank_at_k(relevant, recommended, k):
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1.0 / i
    return 0.0

In [16]:
def evaluate_recommender(recommend_func, users, test_relevant_dict, ks=(5, 10, 20)):
    rows = []
    max_k = max(ks)
    
    for user_id in users:
        relevant = test_relevant_dict.get(user_id, set())
        if not relevant:
            continue
        
        recommended = recommend_func(user_id, top_k=max_k)
        
        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [17]:
tfidf_eval = evaluate_recommender(
    recommend_func=recommend_user_tfidf_ids,
    users=eval_users,
    test_relevant_dict=test_relevant,
    ks=(5, 10, 20)
)

tfidf_eval.head()

,UserID,Precision@5,Recall@5,NDCG@5,MAP@5,Precision@10,Recall@10,NDCG@10,MAP@10,Precision@20,Recall@20,NDCG@20,MAP@20
0,1000257,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1000308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1000420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1000515,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1000554,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
tfidf_summary = tfidf_eval.drop(columns="UserID").mean().round(4).to_frame(name="TF-IDF")
tfidf_summary

,TF-IDF
Precision@5,0.0042
Recall@5,0.0075
NDCG@5,0.0069
MAP@5,0.0044
Precision@10,0.0037
Recall@10,0.0123
NDCG@10,0.0088
MAP@10,0.0049
Precision@20,0.0027
Recall@20,0.0184


Although it seems weird at first sight that the metrics for some of the users are all zeros, when we look at the results globally we observe another scenario. 

In [19]:
popular_ranking = train_pos["WineID"].value_counts().index.tolist()

def recommend_popular_ids(user_id, top_k=10):
    seen_items = train_seen.get(user_id, set())
    recs = [wine_id for wine_id in popular_ranking if wine_id not in seen_items]
    return recs[:top_k]

In [20]:
popular_eval = evaluate_recommender(
    recommend_func=recommend_popular_ids,
    users=eval_users,
    test_relevant_dict=test_relevant,
    ks=(5, 10, 20)
)

popular_summary = popular_eval.drop(columns="UserID").mean().round(4).to_frame(name="Popularity")
popular_summary

,Popularity
Precision@5,0.0044
Recall@5,0.0073
NDCG@5,0.0068
MAP@5,0.0041
Precision@10,0.0038
Recall@10,0.0127
NDCG@10,0.0088
MAP@10,0.0046
Precision@20,0.0033
Recall@20,0.0218


In [21]:
comparison = pd.concat([popular_summary, tfidf_summary], axis=1)
comparison

,Popularity,TF-IDF
Precision@5,0.0044,0.0042
Recall@5,0.0073,0.0075
NDCG@5,0.0068,0.0069
MAP@5,0.0041,0.0044
Precision@10,0.0038,0.0037
Recall@10,0.0127,0.0123
NDCG@10,0.0088,0.0088
MAP@10,0.0046,0.0049
Precision@20,0.0033,0.0027
Recall@20,0.0218,0.0184


In [22]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
tfidf_summary.to_csv("../data/results/tfidf_summary.csv")

In [23]:
wines_features["eval_signature"] = (
    wines_features["Type"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Country"].astype(str).str.lower().str.strip() + "|" +
    wines_features["RegionName"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Body"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Acidity"].astype(str).str.lower().str.strip()
)

wineid_to_signature = dict(zip(wines_features["WineID"], wines_features["eval_signature"]))

test_relevant_signatures = {
    user_id: {wineid_to_signature[w] for w in wine_ids if w in wineid_to_signature}
    for user_id, wine_ids in test_relevant.items()
}

In [24]:
def recommended_ids_to_signatures(recommended_ids, top_k=None):
    signatures = []
    seen_signatures = set()

    for wine_id in recommended_ids:
        if wine_id not in wineid_to_signature:
            continue

        signature = wineid_to_signature[wine_id]
        if signature in seen_signatures:
            continue

        signatures.append(signature)
        seen_signatures.add(signature)

        if top_k is not None and len(signatures) == top_k:
            break

    return signatures

In [25]:
def evaluate_recommender_soft(recommend_func, users, test_relevant_signatures_dict, ks=(5, 10, 20), oversample_factor=5):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_signatures_dict.get(user_id, set())
        if not relevant:
            continue

        recommended_ids = recommend_func(user_id, top_k=max_k * oversample_factor)
        recommended = recommended_ids_to_signatures(recommended_ids, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Soft-evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [26]:
tfidf_eval_soft = evaluate_recommender_soft(
    recommend_func=recommend_user_tfidf_ids,
    users=eval_users,
    test_relevant_signatures_dict=test_relevant_signatures,
    ks=(5, 10, 20),
    oversample_factor=5
)

tfidf_summary_soft = (
    tfidf_eval_soft
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="TF-IDF Soft")
)

tfidf_summary_soft

Soft-evaluated 500/5000 users
Soft-evaluated 1000/5000 users
Soft-evaluated 1500/5000 users
Soft-evaluated 2000/5000 users
Soft-evaluated 2500/5000 users
Soft-evaluated 3000/5000 users
Soft-evaluated 3500/5000 users
Soft-evaluated 4000/5000 users
Soft-evaluated 4500/5000 users
Soft-evaluated 5000/5000 users


,TF-IDF Soft
Precision@5,0.0752
Recall@5,0.1349
NDCG@5,0.1338
MAP@5,0.0943
HitRate@5,0.3150
MRR@5,0.2099
Precision@10,0.0467
Recall@10,0.1660
NDCG@10,0.1425
MAP@10,0.0953


In [27]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
tfidf_eval_soft.to_csv("../data/results/tfidf_eval_soft.csv", index=False)
tfidf_summary_soft.to_csv("../data/results/tfidf_summary_soft.csv")

# Save the model

In [ ]:
from pathlib import Path
import pandas as pd

ARMS_DIR = Path("../bandits/saved_arms")
ARMS_DIR.mkdir(parents=True, exist_ok=True)

def export_arm_recs(recommend_fn, users, out_csv, top_k=100):
    rows = []
    for uid in users:
        recs = recommend_fn(int(uid), top_k=top_k)
        for r, wid in enumerate(recs, start=1):
            rows.append({"UserID": int(uid), "rank": int(r), "WineID": int(wid)})
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)
    print(f"Saved {len(out_df):,} rows -> {out_csv}")


In [ ]:
export_arm_recs(
    recommend_fn=recommend_user_tfidf_ids,
    users=eval_users,
    out_csv=ARMS_DIR / "content_tfidf_recs.csv",
    top_k=100
)
